<a href="https://colab.research.google.com/github/TimofeyProtasov/diploma/blob/main/days/work_1704.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q triton trl

In [ ]:
import torch
import random
from typing import Optional
from sklearn.model_selection import train_test_split
from dataclasses import dataclass, field
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    PeftConfig
)

In [ ]:
@dataclass
class RAGExperiment:
    dataset_name: str = 'phatvo/hotpotqa-raft'
    oracle_presence_ratio: float = 1.0
    sample_size: int = 10
    test_size_ratio: float = 0.2
    model_name: str = 'Qwen/Qwen2.5-0.5B-Instruct'
    peft_config: Optional[PeftConfig] = field(default_factory=lambda: LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ))
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 2
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    warmup_steps: int = 5
    seed: int = 1


    def __post_init__(self):
        random.seed(self.seed)


    def prepare_data(self):
        dataset: DatasetDict = load_dataset('phatvo/hotpotqa-raft')


        def presence_oracle(example):
            return example['oracle_context'] in example['instruction']


        dataset_with_oracle: Dataset = dataset['train'].filter(lambda ex: presence_oracle(ex))
        dataset_without_oracle: Dataset = dataset['train'].filter(lambda ex: not presence_oracle(ex))

        num_with: int = round(self.sample_size * self.oracle_presence_ratio)
        num_without: int = self.sample_size - num_with

        indices_with: list = random.sample(range(len(dataset_with_oracle)), num_with)
        indices_without: list = random.sample(range(len(dataset_without_oracle)), num_without)

        if indices_with:
            idx_train_with_oracle, idx_test_with_oracle = train_test_split(
                indices_with,
                test_size=self.test_size_ratio,
                shuffle=False,
                random_state=self.seed
            )
        else:
            idx_train_with_oracle, idx_test_with_oracle = [], []

        if indices_without:
            idx_train_without_oracle, idx_test_without_oracle = train_test_split(
                indices_without,
                test_size=self.test_size_ratio,
                shuffle=False,
                random_state=self.seed
            )
        else:
            idx_train_without_oracle, idx_test_without_oracle = [], []

        def get_shuffle_dataset(ids_with, ids_without):
            parts = []
            if ids_with:
                parts.append(dataset_with_oracle.select(ids_with))
            if ids_without:
                parts.append(dataset_without_oracle.select(ids_without))
            if parts:
                return concatenate_datasets(parts).shuffle(seed=self.seed)
            else:
                # Возвращаем пустой Dataset (можно создать из пустого словаря)
                return Dataset.from_dict({})


        self.train_dataset: Dataset = get_shuffle_dataset(idx_train_with_oracle, idx_train_without_oracle)
        self.test_dataset: Dataset = get_shuffle_dataset(idx_test_with_oracle, idx_test_without_oracle)


    def prepare_model(self):
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            dtype=torch.bfloat16,
            device_map='auto'
        )


    def prepare_tokenizer(self):
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name
        )


    def format_data(self):
        """Форматируем данные в prompt/completion для SFTTrainer."""
        def format_sft_example(example):
            prompt_messages = [{"role": "user", "content": example["instruction"]}]
            prompt = self.tokenizer.apply_chat_template(
                prompt_messages,
                tokenize=False,
                add_generation_prompt=True
            )
            completion = example["cot_answer"]
            return {"prompt": prompt, "completion": completion}


        self.formatted_train_dataset = self.train_dataset.map(
            format_sft_example,
            remove_columns=self.train_dataset.column_names
        )


    def add_peft_if_exist(self):
        self.model = get_peft_model(self.model, self.peft_config) if self.peft_config else self.model
        self.model = dispatch_model(self.model, device_map="auto")


    def prepare_training(self):
        self.training_args = SFTConfig(
            output_dir="./qwen-raft-lora",
            num_train_epochs=self.num_train_epochs,
            per_device_train_batch_size=self.per_device_train_batch_size,
            gradient_accumulation_steps=self.gradient_accumulation_steps,
            learning_rate=self.learning_rate,
            warmup_steps=self.warmup_steps,
            logging_steps=10,
            save_steps=100,
            save_total_limit=2,
            bf16=True,
            report_to="none",
            remove_unused_columns=False,

            max_length=1024,
            packing=False,
            completion_only_loss=True,
        )

        # Конструктор SFTTrainer теперь принимает только базовые сущности
        self.trainer = SFTTrainer(
            model=self.model,
            args=self.training_args,
            train_dataset=self.formatted_train_dataset,
            processing_class=self.tokenizer,
        )


    def train(self):
        self.trainer.train()


    def evaluate_perplexity(self):
        pass


    def evaluate_f1(self):
        pass


    def run(self):
        self.prepare_data()
        self.prepare_model()
        self.prepare_tokenizer()
        self.format_data()
        self.add_peft_if_exist()
        self.prepare_training()
        self.train()
        self.evaluate_perplexity()
        self.evaluate_f1()
        torch.cuda.empty_cache()

In [ ]:
exp = RAGExperiment(
    dataset_name='phatvo/hotpotqa-raft',
    oracle_presence_ratio=1.0,
    sample_size=10,
    test_size_ratio=0.2,
    model_name='Qwen/Qwen2.5-0.5B-Instruct',
    peft_config=LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=5,
    seed=1
)

In [ ]:
exp.run()